# Stent skeletonisation — class API

Extracts the 1D wireframe (skeleton) of a stent design from its 3D STL surface
mesh, using the `Stent` class from the object-oriented package under `src_oop/`.

`Stent(...).skeletonize()` runs the whole thing; the cells below it show the
same run split into its three phases, which is what you want when a ring needs
manual fixing. Every stage writes an inspectable intermediate (CSV + Plotly
HTML) into the output folder.


In [ ]:
# `src_oop/stentfit` and the old `src/stentfit` share the import name `stentfit`,
# so put src_oop first on the path to pick up the class API. Walk up from the
# working directory to find the repo, then check what actually got imported —
# an installed copy of the old package would otherwise shadow it silently.
import sys
from pathlib import Path


def find_repo_root(start=None):
    """Nearest ancestor directory that contains `src_oop/stentfit`."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src_oop" / "stentfit").is_dir():
            return candidate
    raise RuntimeError(f"no src_oop/stentfit found above {start}")


REPO = find_repo_root()
sys.path.insert(0, str(REPO / "src_oop"))

import stentfit
from stentfit import Stent

assert (REPO / "src_oop") in Path(stentfit.__file__).resolve().parents, (
    f"imported the wrong stentfit: {stentfit.__file__}")
print(f"stentfit {stentfit.__version__} from {Path(stentfit.__file__).parent}")


In [ ]:
STENT_NAME = "stent01"
STL_FILE   = REPO / "examples/data/input/stent_designs" / f"{STENT_NAME}.stl"
OUTPUT_DIR = REPO / "examples/data/output/stent_skeleton_oop" / STENT_NAME

# Tuning parameters are constructor arguments: set once, held on the object.
stent = Stent(
    stl_file=str(STL_FILE),
    stent_name=STENT_NAME,
    output_dir=str(OUTPUT_DIR),
    random_seed=0,
    remove_supports=(STENT_NAME == "16crownCrimpedXienceStent+extremaSupports"),
    auto_tune=True,
    pixels_per_strut=10,
    dilate_px=3,
    tune_time_limit=120,
)
stent


## Run the whole pipeline

One call: sample → detect rings → 2D-skeletonise → (prompt for manual edits) →
wrap to 3D → clean the graph → fit splines. It returns the same object, so you
can keep working with `stent` afterwards.


In [ ]:
stent.skeletonize()


## Or: phase by phase

Use this when you want to inspect `skeleton_plots/ring_XX.html` and fix a defect
by hand before committing to the 3D wrap. Each phase returns `self`, and the
`ring_2d.pkl` checkpoint written by the first one means a kernel restart costs
you nothing.

```python
stent = Stent(str(STL_FILE), STENT_NAME, str(OUTPUT_DIR))
stent.skeletonize_2d()      # sample -> rings -> per-ring 2D skeleton + checkpoint
stent.edit_and_assemble()   # manual 2D fixes -> one assembled flat skeleton
stent.finalize()            # wrap to 3D -> clean graph -> fit splines

# after a kernel restart, pick straight back up from the checkpoint:
stent = Stent.load(str(OUTPUT_DIR))
stent.edit_and_assemble().finalize()
```


## Results

The fitted splines are what the simulation setup consumes; the graph and the
raw point cloud stay available on the object for inspection.


In [ ]:
print(f"skeleton nodes : {len(stent.skeleton_df):,}")
print(f"curves         : {len(stent.skeleton_curves)}")
print(f"splines fitted : {sum(s is not None for s in stent.skeleton_splines)}")
print(f"circumference  : {stent.circumference:.3f} mm  "
      f"(derived from r_mid = {stent.stent_features['r_mid']:.3f} mm)")
print(f"outputs        : {stent.output_dir}")

stent.skeleton_df.head()
